In [79]:
import pandas as pd 
import pandas as pd
import os
import glob
import time

In [80]:
folder_path = r"C:\Users\amin_\Documents\Git Repositories\RoboGardenAK\TA Omar\ml-latest-small"
rating_csvfile = glob.glob(os.path.join(folder_path, "ratings.csv"))
df_ratings=pd.read_csv(rating_csvfile[0])

movies_csvfile = glob.glob(os.path.join(folder_path, "movies.csv"))
df_movies=pd.read_csv(movies_csvfile [0])

tags_csvfile = glob.glob(os.path.join(folder_path, "tags.csv"))
df_tags=pd.read_csv(tags_csvfile [0])

In [81]:
df_tags.head()


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [ ]:
tags_grouped = df_tags.groupby('movieId')['tag'].apply(lambda x: ','.join(x)).reset_index()

# Merge tags with movies
movies_tag = pd.merge(df_movies, tags_grouped, on='movieId', how='left')
movies_tag['tag'] = movies_tag['tag'].fillna('')

movies_tag['combined_features'] = movies_tag['genres'] + ' ' + movies_tag['tag']

# Merge rate with movies_tag
movies_tag_rate = pd.merge(df_ratings,movies_tag,  on='movieId', how='left')

#calculate age in year and add a new column weighted_rating
movies_tag_rate['age_year']= round((round(time.time() )- movies_tag_rate['timestamp'])/(24*3600*365))
movies_tag_rate['weighted_rating'] = movies_tag_rate['rating'] *1/ movies_tag_rate['age_year']

# Convert timestamps
#movies_tag_rate['timestamp'] = pd.to_datetime(movies_tag_rate['timestamp'], unit='s')

movies_tag_rate.head()





,userId,movieId,rating,timestamp,title,genres,tag,combined_features,age_year,weighted_rating
0,1,1,4.0,2000-07-30 18:45:03,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,"pixar,pixar,fun",Adventure|Animation|Children|Comedy|Fantasy pi...,25.0,0.16
1,1,3,4.0,2000-07-30 18:20:47,Grumpier Old Men (1995),Comedy|Romance,"moldy,old","Comedy|Romance moldy,old",25.0,0.16
2,1,6,4.0,2000-07-30 18:37:04,Heat (1995),Action|Crime|Thriller,,Action|Crime|Thriller,25.0,0.16
3,1,47,5.0,2000-07-30 19:03:35,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,"mystery,twist ending,serial killer","Mystery|Thriller mystery,twist ending,serial k...",25.0,0.20
4,1,50,5.0,2000-07-30 18:48:51,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,"mindfuck,suspense,thriller,tricky,twist ending...","Crime|Mystery|Thriller mindfuck,suspense,thril...",25.0,0.20


In [98]:
# Calculate the mean of weighted_rating for each movie and sort it
movies_tag_rate.groupby("movieId").agg({
        "weighted_rating": "mean",
        "title": "first"
    }).sort_values(by="weighted_rating", ascending=False).reset_index()



,movieId,weighted_rating,title
0,172875,0.714286,A Detective Story (2003)
1,187717,0.714286,Won't You Be My Neighbor? (2018)
2,33649,0.714286,Saving Face (2004)
3,179135,0.714286,Blue Planet II (2017)
4,136556,0.714286,Kung Fu Panda: Secrets of the Masters (2011)
...,...,...,...
9719,5105,0.023810,Don't Look Now (1973)
9720,7742,0.023810,Baxter (1989)
9721,6967,0.023810,Dead of Night (1945)
9722,5700,0.022727,The Pumaman (1980)


In [78]:
%reset -f